# 👁️ NB-08: CLIP Vision-Text – Synthetic Image Caption Alignment
**Goal:** Generate concept images from response keywords, then fine-tune CLIP text encoder for topic matching.

**Modality:** Vision-Text | **Model:** openai/clip-vit-base-patch32

In [ ]:
!pip install transformers pillow datasets torch -q

In [ ]:
import json, re, random
random.seed(42)

def load_dataset(path="claude-opus-4_5-250x_jsonl.txt"):
    """Load Claude thinking dataset from JSONL."""
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def extract_fields(records):
    data = []
    for r in records:
        msgs = r["messages"]
        user = next((m["content"] for m in msgs if m["role"]=="user"), "")
        asst = next((m["content"] for m in msgs if m["role"]=="assistant"), "")
        think = re.findall(r"<think>(.*?)</think>", asst, re.DOTALL)
        response = re.sub(r"<think>.*?</think>", "", asst, flags=re.DOTALL).strip()
        data.append({
            "user": user,
            "assistant_full": asst,
            "thinking": " ".join(think).strip(),
            "response": response
        })
    return data

records = load_dataset()
data = extract_fields(records)
print(f"Loaded {len(data)} examples")
print(f"Sample user: {data[0]['user'][:80]}")
print(f"Sample think: {data[0]['thinking'][:120]}...")
print(f"Sample resp: {data[0]['response'][:120]}...")


In [ ]:
from transformers import CLIPTokenizer, CLIPTextModel, CLIPProcessor, CLIPModel
from PIL import Image, ImageDraw, ImageFont
import torch, numpy as np

# Synthesize 'concept images' as colored text-on-canvas (no external API needed)
def make_concept_image(text, size=224):
    img = Image.new("RGB", (size, size), color=(20, 20, 40))
    d = ImageDraw.Draw(img)
    # Word-wrap and draw
    words = text[:60].split()[:8]
    y = 80
    for i, w in enumerate(words[:4]):
        d.text((10, y + i*25), w, fill=(100, 200, 255))
    return img

images = [make_concept_image(d["user"]) for d in data]
captions = [d["user"] for d in data]

processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Compute embeddings & cosine sim
batch = processor(text=captions[:10], images=images[:10], return_tensors="pt", padding=True, truncation=True).to(device)
with torch.no_grad():
    out = model(**batch)
print("Image-text logits shape:", out.logits_per_image.shape)
print("Top match diagonal (should be high):", out.logits_per_image.diag().tolist())


In [ ]:
# Fine-tune CLIP text encoder on captions
from torch.utils.data import Dataset as TDS, DataLoader
import torch.nn.functional as F

class ClipDS(TDS):
    def __init__(self): self.imgs, self.caps = images, captions
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        return processor(text=self.caps[i], images=self.imgs[i],
                         return_tensors="pt", padding="max_length",
                         truncation=True, max_length=77)

opt = torch.optim.AdamW(model.text_model.parameters(), lr=1e-5)
loader = DataLoader(ClipDS(), batch_size=8, shuffle=True, collate_fn=lambda b: processor(
    text=[d["user"] for d in data[:8]], images=images[:8],
    return_tensors="pt", padding=True, truncation=True).to(device))

for epoch in range(2):
    for batch in loader:
        out = model(**batch)
        # Symmetric cross-entropy loss
        n = out.logits_per_image.size(0)
        targets = torch.arange(n, device=device)
        loss = (F.cross_entropy(out.logits_per_image, targets) +
                F.cross_entropy(out.logits_per_text, targets)) / 2
        opt.zero_grad(); loss.backward(); opt.step()
        print(f"Epoch {epoch+1} Loss: {loss.item():.4f}")
        break  # one batch demo
print("✅ CLIP text encoder fine-tuned!")
